In [ ]:
######   preprocessing
import os
import pandas as pd

# ----------- PATHS -------------
input_folder = "../../data/raw/processed_data"
output_folder = "../../data/raw/lstm_data"

os.makedirs(output_folder, exist_ok=True)
# --------------------------------

for file in os.listdir(input_folder):
    if file.endswith(".csv"):
        input_path = os.path.join(input_folder, file)

        # Load csv
        df = pd.read_csv(input_path)

        # Fill only Unit1 & Unit2 with -1
        if "Unit1" in df.columns:
            df["Unit1"] = df["Unit1"].fillna(-1)

        if "Unit2" in df.columns:
            df["Unit2"] = df["Unit2"].fillna(-1)

        # Save cleaned version to output directory
        output_path = os.path.join(output_folder, file)
        df.to_csv(output_path, index=False)

        print(f"Cleaned and saved: {file}")

print("✅ ALL FILES PROCESSED & SAVED IN data/lstm/")


In [ ]:
####### find max sequences length
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

data_dir = "../../data/raw/lstm_data/"

sequences = []
seq_lengths = []

for file in os.listdir(data_dir):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(data_dir, file))

        # Convert to numpy array
        seq = df.values.astype(float)

        sequences.append(seq)
        seq_lengths.append(len(seq))

MAX_LEN = max(seq_lengths)
print("Max sequence length =", MAX_LEN)


In [ ]:
######## padding
PAD_VALUE = -999

padded_sequences = []
for seq in sequences:
    pad_len = MAX_LEN - len(seq)
    if pad_len > 0:
        pad = np.full((pad_len, seq.shape[1]), PAD_VALUE)
        seq = np.vstack([seq, pad])
    padded_sequences.append(seq)

padded_sequences = np.array(padded_sequences)
print("Shape after padding:", padded_sequences.shape)


In [ ]:
########## scaling
IGNORE_COLS = [0, 1]    # unit1, unit2
scaler = StandardScaler()

# 1. Reshape to 2D for scaling
N, T, F = padded_sequences.shape
flat = padded_sequences.reshape(-1, F)

# 2. Identify normal (non-pad) rows
valid_mask = (flat[:, 0] != PAD_VALUE)

# 3. Select only valid rows, and only non-ignore columns
cols_to_scale = [i for i in range(F) if i not in IGNORE_COLS]

valid_data = flat[valid_mask][:, cols_to_scale]

# 4. Fit scaler
scaler.fit(valid_data)

# 5. Transform ONLY valid rows, ONLY selected columns
flat_scaled = flat.copy()
flat_scaled[valid_mask][:, cols_to_scale] = scaler.transform(valid_data)

# 6. Restore padded rows unchanged
flat_scaled[~valid_mask] = PAD_VALUE

# 7. Reshape back to (patients, max_len, features)
scaled_sequences = flat_scaled.reshape(N, T, F)


In [ ]:
import sys
print(sys.version)
!pip install tensorflow

In [ ]:
import tensorflow
print(tensorflow.__version__)

In [ ]:
labels = []

for file in os.listdir(data_dir):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(data_dir, file))
        
        # Assuming the CSV has a column 'SepsisLabel' with 0/1 at each timestep
        # Take the max: 1 if patient ever got sepsis, else 0
        label = df['SepsisLabel'].max()
        labels.append(label)

labels = np.array(labels)
print("Labels shape:", labels.shape)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking

X = scaled_sequences
y = np.array(labels)   # you must supply patient-level labels

model = Sequential([
    Masking(mask_value=PAD_VALUE, input_shape=(MAX_LEN, F)),
    LSTM(64, return_sequences=False),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


In [ ]:
print("Sequences shape:", X.shape)   # (num_patients, MAX_LEN, F)
print("Labels shape:", y.shape)      # (num_patients,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=100, stratify=y
)


In [ ]:
import numpy as np
print("Train class distribution:", np.bincount(y_train))
print("Test class distribution:", np.bincount(y_test))


In [ ]:
import numpy as np

X_minority = X_train[y_train==1]
y_minority = y_train[y_train==1]

X_majority = X_train[y_train==0]
y_majority = y_train[y_train==0]

# Simple duplication
rep_factor = len(y_majority) // len(y_minority)
X_minority_dup = np.tile(X_minority, (rep_factor, 1, 1))
y_minority_dup = np.tile(y_minority, rep_factor)

# Combine
X_train_bal = np.vstack([X_majority, X_minority_dup])
y_train_bal = np.hstack([y_majority, y_minority_dup])

# Shuffle
idx = np.arange(len(y_train_bal))
np.random.shuffle(idx)
X_train_bal = X_train_bal[idx]
y_train_bal = y_train_bal[idx]


In [ ]:
import numpy as np

print("Any NaNs in X_train_bal?", np.isnan(X_train_bal).any())
print("Max value:", np.max(X_train_bal))
print("Min value:", np.min(X_train_bal))


In [ ]:
import numpy as np

# Check which sequences contain NaNs
nan_mask = np.isnan(X_train_bal).any(axis=(1,2))   # True for sequences with any NaN
print("Number of sequences with NaNs:", np.sum(nan_mask))

# Optionally, see indices
nan_indices = np.where(nan_mask)[0]
print("Indices of sequences with NaNs:", nan_indices)


In [ ]:
X_train_bal = np.nan_to_num(X_train_bal, nan=PAD_VALUE)
X_test = np.nan_to_num(X_test, nan=PAD_VALUE)


In [ ]:
print("Any NaNs in X_train_bal?", np.isnan(X_train_bal).any())
print("Any NaNs in X_test?", np.isnan(X_test).any())


In [ ]:
###### shuffling coz we padded
indices = np.arange(len(y_train_bal))
np.random.shuffle(indices)
X_train_bal = X_train_bal[indices]
y_train_bal = y_train_bal[indices]


In [ ]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import LSTM, Dense, Masking, Dropout
# from tensorflow.keras.optimizers import Adam

# model = Sequential([
#     Masking(mask_value=PAD_VALUE, input_shape=(MAX_LEN, F)),
#     LSTM(64, return_sequences=False),
#     Dropout(0.3), # 0.2 # helps with stability and overfitting
#     Dense(1, activation='sigmoid')
# ])

# optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
# model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# --- Build LSTM model ---
model = Sequential([
    Masking(mask_value=PAD_VALUE, input_shape=(MAX_LEN, F)),
    LSTM(64, return_sequences=False, dropout=0.3, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])

# --- Compile ---
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])


In [ ]:
# history = model.fit(
#     X_train_bal, y_train_bal,
#     validation_split=0.2,
#     epochs=50, ## 20
#     batch_size=64 ## 32
# )
es = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# --- Train ---
history = model.fit(
    X_train_bal, y_train_bal,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[es],
    shuffle=True
)

# --- Evaluate ---
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))
print("Accuracy:", accuracy_score(y_test, y_pred))


In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))
print("Accuracy:", accuracy_score(y_test, y_pred))


In [ ]:
model.save("lstm_new_sepsis_model.h5")


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
prediction_horizon = 6  # predict sepsis 6 hours before onset


In [ ]:
####### adjust sequences and labels for early prediction
import os
import pandas as pd
import numpy as np
early_X_sequences = []
early_labels = []
data_dir = "../../data/raw/lstm_data/"
for file in os.listdir(data_dir):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(data_dir, file))

        if df["SepsisLabel"].sum() > 0:
            onset = df[df["SepsisLabel"]==1].index[0]
            early_idx = max(0, onset - prediction_horizon)

            seq = df.iloc[:early_idx].values.astype(float)
            label = 1
        else:
            seq = df.values.astype(float)
            label = 0

        early_X_sequences.append(seq)
        early_labels.append(label)

early_labels = np.array(early_labels)


In [ ]:
# ------------------ Step 4: Train/Test Split, Scaling, Class Weights ------------------

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
PAD_VALUE = -999
# Split by patient (stratify on labels)
X_train, X_test, y_train, y_test = train_test_split(
    X_early, y_early, test_size=0.25, random_state=100, stratify=y_early
)

# Scaling (ignore Unit1 & Unit2 columns)
IGNORE_COLS = [0, 1]
N, T, F = X_train.shape
flat = X_train.reshape(-1, F)
valid_mask = (flat[:, 0] != PAD_VALUE)
cols_to_scale = [i for i in range(F) if i not in IGNORE_COLS]

scaler = StandardScaler()
scaler.fit(flat[valid_mask][:, cols_to_scale])
flat[valid_mask][:, cols_to_scale] = scaler.transform(flat[valid_mask][:, cols_to_scale])
X_train = flat.reshape(N, T, F)

# Apply same scaler to test set
flat_test = X_test.reshape(-1, F)
valid_mask_test = (flat_test[:, 0] != PAD_VALUE)
flat_test[valid_mask_test][:, cols_to_scale] = scaler.transform(flat_test[valid_mask_test][:, cols_to_scale])
X_test = flat_test.reshape(X_test.shape[0], T, F)

# ------------------ Apply Class Weights (No Manual Duplication) ------------------

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {0: class_weights_arr[0], 1: class_weights_arr[1]}
print("Class weights:", class_weight_dict)

# Now use X_train and y_train directly for model training


In [ ]:
print("NaNs in X_train:", np.isnan(X_train).sum())
print("NaNs in X_test:", np.isnan(X_test).sum())
print("NaNs in y_train:", np.isnan(y_train).sum())


In [ ]:
# Replace NaNs with PAD_VALUE

import numpy as np
X_train = np.nan_to_num(X_train, nan=PAD_VALUE, posinf=PAD_VALUE, neginf=PAD_VALUE)
X_test = np.nan_to_num(X_test, nan=PAD_VALUE, posinf=PAD_VALUE, neginf=PAD_VALUE)


In [ ]:

# ------------------ Step 5: LSTM Model ------------------

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

model_early = Sequential([
    Masking(mask_value=-999, input_shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(64, return_sequences=False, dropout=0.3, recurrent_dropout=0),
    Dense(1, activation='sigmoid')
])

def focal_loss(gamma=.2, alpha=.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
        p_t = y_true*y_pred + (1-y_true)*(1-y_pred)
        alpha_factor = y_true*alpha + (1-y_true)*(1-alpha)
        modulating_factor = tf.pow(1.0 - p_t, gamma)
        return tf.reduce_mean(alpha_factor * modulating_factor * bce)
    return loss
def recall_m(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    true_positives = tf.reduce_sum(y_true * y_pred)
    possible_positives = tf.reduce_sum(y_true)
    return true_positives / (possible_positives + tf.keras.backend.epsilon())

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model_early.compile(loss=focal_loss(), optimizer=optimizer, metrics=['accuracy', recall_m])

# Early stopping to save time
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train with fewer epochs for faster run
history = model_early.fit(
    X_train_aug, y_train_aug,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    class_weight=class_weight_dict,  # still here
    callbacks=[es],
    shuffle=True,
    verbose=1
)


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# 1️⃣ Predict probabilities
y_pred_prob = model_early.predict(X_test)

# 2️⃣ Convert probabilities to binary labels (0/1) using 0.5 threshold
y_pred = (y_pred_prob > 0.3).astype(int)

# 3️⃣ Print classification metrics
print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

# 4️⃣ ROC-AUC score
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("ROC-AUC:", roc_auc)

# 5️⃣ Accuracy
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)


In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# 1️⃣ Get false positive rate, true positive rate, thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

# 2️⃣ Compute AUC
roc_auc = auc(fpr, tpr)

# 3️⃣ Plot ROC curve
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Guess')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import recall_score

# Early Prediction Recall = recall on the modified early-detection labels
early_recall = recall_score(y_test, y_pred)

print("Early Prediction Recall @ 6 hours:", early_recall)
